### Импорты

In [26]:
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset
from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

import torch
from transformers import AutoTokenizer, AutoModel

### Воспроизводимость и устройство

In [27]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


### Общие функции

In [28]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def train_word2vec_model(token_lists, config):
    model = Word2Vec(
        sentences=token_lists,
        vector_size=config["vector_size"],
        window=config["window"],
        min_count=config["min_count"],
        sg=config["sg"],
        negative=config["negative"],
        epochs=config["epochs"],
        workers=4,
        seed=SEED,
    )
    return model

def average_embedding(tokens, keyed_vectors):
    vectors = [keyed_vectors[w] for w in tokens if w in keyed_vectors]
    if len(vectors) == 0:
        return np.zeros(keyed_vectors.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0)

def texts_to_w2v_features(token_lists, keyed_vectors):
    return np.vstack([average_embedding(tokens, keyed_vectors) for tokens in token_lists])

def evaluate_classifier(X_train, y_train, X_test, y_test, dataset_name, feature_name):
    clf = LogisticRegression(max_iter=3000, random_state=SEED)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    return {
        "dataset": dataset_name,
        "features": feature_name,
        "f1_macro": f1_score(y_test, pred, average="macro"),
        "f1_micro": f1_score(y_test, pred, average="micro"),
        "f1_weighted": f1_score(y_test, pred, average="weighted"),
        "classifier": "LogisticRegression"
    }, clf, pred

def build_nearest_words_table(keyed_vectors, query_words, topn=10):
    rows = []
    for word in query_words:
        if word not in keyed_vectors:
            continue
        sims = keyed_vectors.most_similar(word, topn=topn)
        row = {"query_word": word}
        for i, (sim_word, score) in enumerate(sims, start=1):
            row[f"top_{i}"] = sim_word
            row[f"sim_{i}"] = round(float(score), 4)
        rows.append(row)
    return pd.DataFrame(rows)

def select_query_words(preferred_words, keyed_vectors, n=10):
    selected = [w for w in preferred_words if w in keyed_vectors]
    if len(selected) < n:
        for w in keyed_vectors.index_to_key:
            if w not in selected and w.isalpha() and len(w) > 3:
                selected.append(w)
            if len(selected) == n:
                break
    return selected[:n]

def bert_cls_mean_embeddings(texts, tokenizer, model, batch_size=16, max_length=128, device=device):
    cls_all = []
    mean_all = []

    model.eval()
    for start in tqdm(range(0, len(texts), batch_size), desc="BERT embeddings"):
        batch_texts = texts[start:start+batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc)
            hidden = out.last_hidden_state  # [B, T, H]

            cls_emb = hidden[:, 0, :]

            mask = enc["attention_mask"].unsqueeze(-1)  # [B, T, 1]
            mean_emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        cls_all.append(cls_emb.cpu().numpy())
        mean_all.append(mean_emb.cpu().numpy())

    return np.vstack(cls_all), np.vstack(mean_all)

### Конфигурации Word2Vec

In [29]:
w2v_configs = [
    {
        "name": "cbow_100_w5",
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "sg": 0,          
        "negative": 5,
        "epochs": 10,
    },
    {
        "name": "skipgram_100_w5",
        "vector_size": 100,
        "window": 5,
        "min_count": 2,
        "sg": 1,        
        "negative": 5,
        "epochs": 10,
    },
    {
        "name": "skipgram_200_w8",
        "vector_size": 200,
        "window": 8,
        "min_count": 2,
        "sg": 1,
        "negative": 10,
        "epochs": 15,
    },
]
w2v_configs

[{'name': 'cbow_100_w5',
  'vector_size': 100,
  'window': 5,
  'min_count': 2,
  'sg': 0,
  'negative': 5,
  'epochs': 10},
 {'name': 'skipgram_100_w5',
  'vector_size': 100,
  'window': 5,
  'min_count': 2,
  'sg': 1,
  'negative': 5,
  'epochs': 10},
 {'name': 'skipgram_200_w8',
  'vector_size': 200,
  'window': 8,
  'min_count': 2,
  'sg': 1,
  'negative': 10,
  'epochs': 15}]

### BERT-модель

In [30]:
BERT_MODEL_NAME = "bert-base-uncased"

tokenizer_bert = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
model_bert = AutoModel.from_pretrained(BERT_MODEL_NAME).to(device)

print("Loaded:", BERT_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7319.65it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: bert-base-uncased


## Emotion — загрузка и предобработка

In [31]:
dataset_emotion = load_dataset("emotion")

emotion_train_texts = list(dataset_emotion["train"]["text"])
emotion_test_texts = list(dataset_emotion["test"]["text"])

emotion_train_labels = np.array(dataset_emotion["train"]["label"])
emotion_test_labels = np.array(dataset_emotion["test"]["label"])

emotion_train_tokens = [tokenize(t) for t in emotion_train_texts]
emotion_test_tokens = [tokenize(t) for t in emotion_test_texts]

emotion_train_texts_clean = [" ".join(tokens) for tokens in emotion_train_tokens]
emotion_test_texts_clean = [" ".join(tokens) for tokens in emotion_test_tokens]

print(len(emotion_train_texts), len(emotion_test_texts))

16000 2000


## Emotion — часть 1. Word2Vec

### 1. Обучаем несколько конфигураций Word2Vec

In [32]:
emotion_w2v_models = {}
emotion_w2v_results = []

for cfg in w2v_configs:
    print(f"Training Word2Vec: {cfg['name']}")
    model_w2v = train_word2vec_model(emotion_train_tokens, cfg)
    emotion_w2v_models[cfg["name"]] = model_w2v

    X_train_w2v = texts_to_w2v_features(emotion_train_tokens, model_w2v.wv)
    X_test_w2v = texts_to_w2v_features(emotion_test_tokens, model_w2v.wv)

    result_row, clf, pred = evaluate_classifier(
        X_train_w2v,
        emotion_train_labels,
        X_test_w2v,
        emotion_test_labels,
        dataset_name="emotion",
        feature_name=f"Word2Vec_mean::{cfg['name']}"
    )
    emotion_w2v_results.append(result_row)

df_emotion_w2v_results = pd.DataFrame(emotion_w2v_results).sort_values("f1_macro", ascending=False)
df_emotion_w2v_results

Training Word2Vec: cbow_100_w5
Training Word2Vec: skipgram_100_w5
Training Word2Vec: skipgram_200_w8


,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
2,emotion,Word2Vec_mean::skipgram_200_w8,0.402624,0.5745,0.535125,LogisticRegression
1,emotion,Word2Vec_mean::skipgram_100_w5,0.301580,0.5105,0.450944,LogisticRegression
0,emotion,Word2Vec_mean::cbow_100_w5,0.191782,0.4195,0.339769,LogisticRegression


### 2. Выбираем лучшую Word2Vec-конфигурацию и смотрим ближайшие слова

In [33]:
best_emotion_w2v_name = df_emotion_w2v_results.iloc[0]["features"].split("::")[1]
best_emotion_w2v = emotion_w2v_models[best_emotion_w2v_name]

preferred_emotion_words = [
    "happy", "sad", "angry", "love", "fear",
    "surprise", "hope", "hate", "feel", "joy"
]

emotion_query_words = select_query_words(preferred_emotion_words, best_emotion_w2v.wv, n=10)
emotion_nearest_words = build_nearest_words_table(best_emotion_w2v.wv, emotion_query_words, topn=10)

print("Лучшая конфигурация Word2Vec для emotion:", best_emotion_w2v_name)
emotion_nearest_words

Лучшая конфигурация Word2Vec для emotion: skipgram_200_w8


,query_word,top_1,sim_1,top_2,sim_2,top_3,sim_3,top_4,sim_4,top_5,...,top_6,sim_6,top_7,sim_7,top_8,sim_8,top_9,sim_9,top_10,sim_10
0,happy,eternally,0.6350,invaded,0.6317,careful,0.6228,smiley,0.5973,tranquil,...,moody,0.5899,dieting,0.5790,allah,0.5769,needlessly,0.5743,powerless,0.5721
1,sad,despondent,0.6318,angry,0.6002,deceived,0.5996,vunerable,0.5936,ignoring,...,moody,0.5815,embarrased,0.5813,apparently,0.5796,lonely,0.5771,disconnected,0.5760
2,angry,upset,0.6586,aggressive,0.6567,defending,0.6239,moody,0.6223,instinct,...,deceived,0.6068,panicked,0.6063,misunderstood,0.6014,unsatisfied,0.6011,sad,0.6002
3,love,inanimate,0.5088,hate,0.5031,dearly,0.4902,shares,0.4866,budget,...,watchers,0.4840,resent,0.4839,burns,0.4825,cap,0.4811,annoy,0.4805
4,fear,suppressed,0.6899,enjoyment,0.6805,drake,0.6585,optimism,0.6570,refusing,...,amish,0.6435,baggage,0.6395,rape,0.6385,recognizing,0.6383,damaging,0.6382
5,surprise,announced,0.7209,halloween,0.7127,dublin,0.7085,heartbreak,0.7015,thanked,...,whim,0.7009,prom,0.6994,root,0.6976,banks,0.6893,shirts,0.6889
6,hope,eternal,0.5685,advise,0.5565,assure,0.5545,someday,0.5450,request,...,earning,0.5436,noe,0.5428,suggest,0.5366,alex,0.5357,creations,0.5347
7,hate,dislike,0.5954,defending,0.5953,needy,0.5899,unwanted,0.5807,despised,...,bah,0.5793,complain,0.5708,reserved,0.5689,fuss,0.5683,sucks,0.5657
8,feel,vunerable,0.6305,criticized,0.6298,eternally,0.6278,i,0.6202,intentional,...,profoundly,0.6137,horrid,0.6106,worldly,0.6068,guilted,0.6065,soothe,0.6061
9,joy,acceptance,0.6763,treasure,0.6297,bearable,0.6273,truely,0.6091,miracles,...,tia,0.6055,blessings,0.6025,forgiveness,0.5986,cherish,0.5981,attain,0.5980


## Emotion — BERT

In [34]:
emotion_train_cls, emotion_train_mean = bert_cls_mean_embeddings(
    emotion_train_texts,
    tokenizer_bert,
    model_bert,
    batch_size=16,
    max_length=64,
    device=device
)

emotion_test_cls, emotion_test_mean = bert_cls_mean_embeddings(
    emotion_test_texts,
    tokenizer_bert,
    model_bert,
    batch_size=16,
    max_length=64,
    device=device
)

BERT embeddings: 100%|██████████| 125/125 [00:56<00:00,  2.23it/s]


In [35]:
emotion_bert_results = []

row_cls, clf_cls, pred_cls = evaluate_classifier(
    emotion_train_cls,
    emotion_train_labels,
    emotion_test_cls,
    emotion_test_labels,
    dataset_name="emotion",
    feature_name="BERT_CLS"
)
emotion_bert_results.append(row_cls)

row_mean, clf_mean, pred_mean = evaluate_classifier(
    emotion_train_mean,
    emotion_train_labels,
    emotion_test_mean,
    emotion_test_labels,
    dataset_name="emotion",
    feature_name="BERT_mean"
)
emotion_bert_results.append(row_mean)

df_emotion_bert_results = pd.DataFrame(emotion_bert_results).sort_values("f1_micro", ascending=False)
df_emotion_bert_results

,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
1,emotion,BERT_mean,0.558864,0.6515,0.642992,LogisticRegression
0,emotion,BERT_CLS,0.484150,0.6035,0.589959,LogisticRegression


In [36]:
df_emotion_compare = pd.concat(
    [df_emotion_w2v_results, df_emotion_bert_results],
    ignore_index=True
).sort_values("f1_micro", ascending=False)

df_emotion_compare

,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
3,emotion,BERT_mean,0.558864,0.6515,0.642992,LogisticRegression
4,emotion,BERT_CLS,0.484150,0.6035,0.589959,LogisticRegression
0,emotion,Word2Vec_mean::skipgram_200_w8,0.402624,0.5745,0.535125,LogisticRegression
1,emotion,Word2Vec_mean::skipgram_100_w5,0.301580,0.5105,0.450944,LogisticRegression
2,emotion,Word2Vec_mean::cbow_100_w5,0.191782,0.4195,0.339769,LogisticRegression


## 20_newsgroups(4) — загрузка и предобработка

In [37]:
dataset_news = load_dataset("SetFit/20_newsgroups")

news_categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

train_news = dataset_news["train"].filter(lambda example: example["label_text"] in news_categories)
test_news = dataset_news["test"].filter(lambda example: example["label_text"] in news_categories)

news_label2id = {label: i for i, label in enumerate(news_categories)}

news_train_texts = list(train_news["text"])
news_test_texts = list(test_news["text"])

news_train_labels = np.array([news_label2id[x] for x in train_news["label_text"]])
news_test_labels = np.array([news_label2id[x] for x in test_news["label_text"]])

news_train_tokens = [tokenize(t) for t in news_train_texts]
news_test_tokens = [tokenize(t) for t in news_test_texts]

news_train_texts_clean = [" ".join(tokens) for tokens in news_train_tokens]
news_test_texts_clean = [" ".join(tokens) for tokens in news_test_tokens]

Repo card metadata block was not found. Setting CardData to empty.


## 20_newsgroups(4) — Word2Vec

In [38]:
news_w2v_models = {}
news_w2v_results = []

for cfg in w2v_configs:
    print(f"Training Word2Vec: {cfg['name']}")
    model_w2v = train_word2vec_model(news_train_tokens, cfg)
    news_w2v_models[cfg["name"]] = model_w2v

    X_train_w2v = texts_to_w2v_features(news_train_tokens, model_w2v.wv)
    X_test_w2v = texts_to_w2v_features(news_test_tokens, model_w2v.wv)

    result_row, clf, pred = evaluate_classifier(
        X_train_w2v,
        news_train_labels,
        X_test_w2v,
        news_test_labels,
        dataset_name="20_newsgroups(4)",
        feature_name=f"Word2Vec_mean::{cfg['name']}"
    )
    news_w2v_results.append(result_row)

df_news_w2v_results = pd.DataFrame(news_w2v_results).sort_values("f1_micro", ascending=False)
df_news_w2v_results

Training Word2Vec: cbow_100_w5
Training Word2Vec: skipgram_100_w5
Training Word2Vec: skipgram_200_w8


,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
2,20_newsgroups(4),Word2Vec_mean::skipgram_200_w8,0.755116,0.754644,0.755360,LogisticRegression
1,20_newsgroups(4),Word2Vec_mean::skipgram_100_w5,0.709322,0.709161,0.709655,LogisticRegression
0,20_newsgroups(4),Word2Vec_mean::cbow_100_w5,0.623258,0.623318,0.623756,LogisticRegression


### 10 слов и ближайшие соседи для лучшей Word2Vec-модели

In [39]:
best_news_w2v_name = df_news_w2v_results.iloc[0]["features"].split("::")[1]
best_news_w2v = news_w2v_models[best_news_w2v_name]

preferred_news_words = [
    "computer", "graphics", "windows", "image", "file",
    "drive", "card", "screen", "mac", "software"
]

news_query_words = select_query_words(preferred_news_words, best_news_w2v.wv, n=10)
news_nearest_words = build_nearest_words_table(best_news_w2v.wv, news_query_words, topn=10)

print("Лучшая конфигурация Word2Vec для 20_newsgroups(4):", best_news_w2v_name)
news_nearest_words

Лучшая конфигурация Word2Vec для 20_newsgroups(4): skipgram_200_w8


,query_word,top_1,sim_1,top_2,sim_2,top_3,sim_3,top_4,sim_4,top_5,...,top_6,sim_6,top_7,sim_7,top_8,sim_8,top_9,sim_9,top_10,sim_10
0,computer,science,0.4967,aided,0.4861,electrical,0.4683,verlag,0.4590,springer,...,retailers,0.4502,scientist,0.4485,amann,0.4449,professor,0.4418,shopper,0.4341
1,graphics,bibliography,0.5224,raytracing,0.5017,gems,0.5008,springer,0.4976,searches,...,specialized,0.4911,modex,0.4821,verlag,0.4683,devoted,0.4598,grfwk,0.4588
2,windows,nt,0.5301,xenix,0.5075,seamless,0.5016,apps,0.4990,workgroups,...,followups,0.4882,crap,0.4857,desqview,0.4839,advocacy,0.4781,dpmi,0.4780
3,image,processing,0.6743,painting,0.5293,enhancement,0.5237,analyst,0.5224,ihs,...,blur,0.5119,decompresses,0.5109,crop,0.5100,smoothing,0.5088,aforementioned,0.5079
4,file,xinto,0.5320,naming,0.5194,uuprog,0.5179,xplace,0.5163,directory,...,builderxcessory,0.5041,hqx,0.5023,xin,0.5000,uubuild,0.4993,fopen,0.4991
5,drive,connor,0.6499,drives,0.6111,hard,0.5981,partitioned,0.5971,shunt,...,jasmine,0.5947,ribbon,0.5945,miniscribe,0.5938,dataframe,0.5920,quantum,0.5916
6,card,cards,0.6235,cages,0.5640,cirrus,0.5614,gup,0.5580,miro,...,fahrenheit,0.5540,backplane,0.5526,activate,0.5417,localbus,0.5415,paradise,0.5404
7,screen,estate,0.5271,loose,0.5070,horizontally,0.5040,dithering,0.4986,scales,...,metallic,0.4934,dim,0.4931,panning,0.4917,halftone,0.4870,compactvideo,0.4849
8,mac,iix,0.5502,iicx,0.5392,aug,0.5067,om,0.4922,ii,...,se,0.4808,iici,0.4786,expansion,0.4784,macconnection,0.4723,powerbook,0.4650
9,software,tools,0.4560,monitoring,0.4483,imdisp,0.4397,bbss,0.4388,products,...,intergraph,0.4344,claris,0.4343,microstation,0.4308,serialised,0.4307,java,0.4283


## 20_newsgroups(4) — BERT

In [40]:
news_train_cls, news_train_mean = bert_cls_mean_embeddings(
    news_train_texts,
    tokenizer_bert,
    model_bert,
    batch_size=8,
    max_length=128,
    device=device
)

news_test_cls, news_test_mean = bert_cls_mean_embeddings(
    news_test_texts,
    tokenizer_bert,
    model_bert,
    batch_size=8,
    max_length=128,
    device=device
)

BERT embeddings: 100%|██████████| 196/196 [01:58<00:00,  1.65it/s]


In [41]:
news_bert_results = []

row_cls, clf_cls, pred_cls = evaluate_classifier(
    news_train_cls,
    news_train_labels,
    news_test_cls,
    news_test_labels,
    dataset_name="20_newsgroups(4)",
    feature_name="BERT_CLS"
)
news_bert_results.append(row_cls)

row_mean, clf_mean, pred_mean = evaluate_classifier(
    news_train_mean,
    news_train_labels,
    news_test_mean,
    news_test_labels,
    dataset_name="20_newsgroups(4)",
    feature_name="BERT_mean"
)
news_bert_results.append(row_mean)

df_news_bert_results = pd.DataFrame(news_bert_results).sort_values("f1_micro", ascending=False)
df_news_bert_results

,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
1,20_newsgroups(4),BERT_mean,0.674339,0.673927,0.674633,LogisticRegression
0,20_newsgroups(4),BERT_CLS,0.651776,0.650865,0.652064,LogisticRegression


In [42]:
df_news_compare = pd.concat(
    [df_news_w2v_results, df_news_bert_results],
    ignore_index=True
).sort_values("f1_micro", ascending=False)

df_news_compare

,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
0,20_newsgroups(4),Word2Vec_mean::skipgram_200_w8,0.755116,0.754644,0.755360,LogisticRegression
1,20_newsgroups(4),Word2Vec_mean::skipgram_100_w5,0.709322,0.709161,0.709655,LogisticRegression
3,20_newsgroups(4),BERT_mean,0.674339,0.673927,0.674633,LogisticRegression
4,20_newsgroups(4),BERT_CLS,0.651776,0.650865,0.652064,LogisticRegression
2,20_newsgroups(4),Word2Vec_mean::cbow_100_w5,0.623258,0.623318,0.623756,LogisticRegression


## Итоговое сравнение по двум датасетам

In [43]:
final_compare = pd.concat(
    [df_emotion_compare, df_news_compare],
    ignore_index=True
).sort_values(["dataset", "f1_micro"], ascending=[True, False])

final_compare

,dataset,features,f1_macro,f1_micro,f1_weighted,classifier
5,20_newsgroups(4),Word2Vec_mean::skipgram_200_w8,0.755116,0.754644,0.755360,LogisticRegression
6,20_newsgroups(4),Word2Vec_mean::skipgram_100_w5,0.709322,0.709161,0.709655,LogisticRegression
7,20_newsgroups(4),BERT_mean,0.674339,0.673927,0.674633,LogisticRegression
8,20_newsgroups(4),BERT_CLS,0.651776,0.650865,0.652064,LogisticRegression
9,20_newsgroups(4),Word2Vec_mean::cbow_100_w5,0.623258,0.623318,0.623756,LogisticRegression
0,emotion,BERT_mean,0.558864,0.651500,0.642992,LogisticRegression
1,emotion,BERT_CLS,0.484150,0.603500,0.589959,LogisticRegression
2,emotion,Word2Vec_mean::skipgram_200_w8,0.402624,0.574500,0.535125,LogisticRegression
3,emotion,Word2Vec_mean::skipgram_100_w5,0.301580,0.510500,0.450944,LogisticRegression
4,emotion,Word2Vec_mean::cbow_100_w5,0.191782,0.419500,0.339769,LogisticRegression
